In [ ]:
import random
import numpy as np
import json
from models import *
import torch
from sklearn.metrics import roc_auc_score
import torch.nn as nn
device = 'mps'

In [2]:
arr = {}
inputs = {}
targets = {}
for i in range(2, 7):
    test_inputs = np.memmap(f'test_data/inputs_{i}.dat', dtype=np.uint8, mode='r', shape=(20000, i, 128, 128, 3))
    test_targets = np.memmap(f'test_data/targets_{i}.dat', dtype=np.uint8, mode='r', shape=(20000,))
    with open(f"identity/test_{i}.json", "r", encoding="utf-8") as file:
        identity = json.load(file)
        arr[i] = {}
        for j, v in enumerate(identity):
            if v not in arr[i]: arr[i][v] = []
            arr[i][v].append(j)
        inputs[i] = [torch.tensor(test_inputs[arr[i][j]]) for j in arr[i]]
        targets[i] = [torch.tensor(test_targets[arr[i][j]]) for j in arr[i]]

In [ ]:
def bootstrap(n, file):
    global inputs
    global targets
    random.seed(42)
    models = {2: NeuralNet2, 3: NeuralNet3, 4: NeuralNet4, 5: NeuralNet5, 6: NeuralNet6}
    model = models[n]().to(device)
    model1 = NeuralNet2().to(device)
    model.load_state_dict(torch.load(file, map_location=device))
    model1.load_state_dict(torch.load('model_2/epoch_14.pth', map_location=device))
    model.eval()
    model1.eval()
    loss_function = nn.BCEWithLogitsLoss()
    loss_diff = []
    acc_diff = []
    roc_diff = []
    for _ in range(300):
        correct = correct1 = 0
        total_loss = total_loss1 = 0.0
        total = 0
        all_logits = []
        test_targets = []
        order = []
        for i in range(len(inputs[n])):
            order.append(random.randint(0, len(inputs[n]) - 1))
        all_probs = []
        with torch.no_grad():
            for i in order:
                batch_inputs = inputs[n][i].float().to(device) / 255.0
                batch_targets = targets[n][i].float().to(device)
                batch_inputs = [batch_inputs[:, i].permute(0, 3, 1, 2) for i in range(n)]
                logits = model(*batch_inputs).reshape(-1)
                batch_targets = batch_targets.reshape(-1)
                all_logits.append(logits.detach().cpu().numpy())
                test_targets.append(batch_targets.detach().cpu().numpy().reshape(-1))
                loss = loss_function(logits, batch_targets)
                predictions = logits > 0
                correct += (predictions == batch_targets).sum().item()
                total_loss += loss.item() * batch_targets.size(0)
                total += batch_targets.size(0)
                probs = torch.zeros_like(batch_targets)
                for ref_idx in range(n - 1):
                    ref = batch_inputs[ref_idx]
                    query = batch_inputs[-1]
                    probs += torch.sigmoid(model1(ref, query)).reshape(-1)
                probs /= n - 1
                all_probs.append(probs.cpu().numpy())
                loss1 = F.binary_cross_entropy(probs, batch_targets)
                total_loss1 += loss1.item() * batch_targets.size(0)
                predictions = probs > 0.5
                correct1 += (predictions == batch_targets).sum().item()
        y_score = np.concatenate(all_logits)
        y_true = np.concatenate(test_targets)
        roc_auc = roc_auc_score(y_true, y_score)
        y_score1 = np.concatenate(all_probs)
        roc_auc1 = roc_auc_score(y_true, y_score1)
        loss_diff.append(total_loss / total - total_loss1 / total)
        acc_diff.append(correct / total - correct1 / total)
        roc_diff.append(roc_auc - roc_auc1)
        print(f"Bootstrap iteration {_ + 1}: Loss difference: {loss_diff[-1]:.4f}, Accuracy difference: {acc_diff[-1]:.4f}, ROC AUC difference: {roc_diff[-1]:.4f}")
    return loss_diff, acc_diff, roc_diff

In [ ]:
loss3, acc3, roc3 = bootstrap(4, 'model_3/epoch_15.pth')

In [ ]:
loss4, acc4, roc4 = bootstrap(4, 'model_4/epoch_15.pth')

In [ ]:
loss5, acc5, roc5 = bootstrap(4, 'model_5/epoch_14.pth')

In [ ]:
loss6, acc6, roc6 = bootstrap(4, 'model_6/epoch_15.pth')

In [ ]:
mean_loss3, mean_acc3, mean_roc3 = np.mean(loss3), np.mean(acc3), np.mean(roc3)
mean_loss4, mean_acc4, mean_roc4 = np.mean(loss4), np.mean(acc4), np.mean(roc4)
mean_loss5, mean_acc5, mean_roc5 = np.mean(loss5), np.mean(acc5), np.mean(roc5)
mean_loss6, mean_acc6, mean_roc6 = np.mean(loss6), np.mean(acc6), np.mean(roc6)
sd_loss3, sd_acc3, sd_roc3 = np.std(loss3, ddof=1), np.std(acc3, ddof=1), np.std(roc3, ddof=1)
sd_loss4, sd_acc4, sd_roc4 = np.std(loss4, ddof=1), np.std(acc4, ddof=1), np.std(roc4, ddof=1)
sd_loss5, sd_acc5, sd_roc5 = np.std(loss5, ddof=1), np.std(acc5, ddof=1), np.std(roc5, ddof=1)
sd_loss6, sd_acc6, sd_roc6 = np.std(loss6, ddof=1), np.std(acc6, ddof=1), np.std(roc6, ddof=1)

In [ ]:
def bootstrap1(file, type='mds_net'):
    n = 4
    global inputs
    global targets
    random.seed(42)
    model = NeuralNet4().to(device)
    types = {'unsorted': NeuralNet4, 'nofc': NeuralNet4, 'nodist': NeuralNet4, 'avg': NeuralNet4}
    model1 = types[type]().to(device)
    model.load_state_dict(torch.load('model_4/epoch_15.pth', map_location=device))
    model1.load_state_dict(torch.load(file, map_location=device))
    model.eval()
    model1.eval()
    loss_function = nn.BCEWithLogitsLoss()
    loss_diff = []
    acc_diff = []
    roc_diff = []
    for _ in range(300):
        correct = correct1 = 0
        total_loss = total_loss1 = 0.0
        total = 0
        all_logits = []
        all_logits1 = []
        test_targets = []
        order = []
        for i in range(len(inputs[n])):
            order.append(random.randint(0, len(inputs[n]) - 1))
        with torch.no_grad():
            for i in order:
                batch_inputs = inputs[n][i].float().to(device) / 255.0
                batch_targets = targets[n][i].float().to(device)
                batch_inputs = [batch_inputs[:, i].permute(0, 3, 1, 2) for i in range(n)]
                logits = model(*batch_inputs).reshape(-1)
                batch_targets = batch_targets.reshape(-1)
                all_logits.append(logits.detach().cpu().numpy())
                test_targets.append(batch_targets.detach().cpu().numpy().reshape(-1))
                loss = loss_function(logits, batch_targets)
                predictions = logits > 0
                correct += (predictions == batch_targets).sum().item()
                total_loss += loss.item() * batch_targets.size(0)
                total += batch_targets.size(0)
                logits1 = model1(*batch_inputs).reshape(-1)
                all_logits1.append(logits1.detach().cpu().numpy())
                loss1 = loss_function(logits1, batch_targets)
                predictions1 = logits1 > 0
                correct1 += (predictions1 == batch_targets).sum().item()
                total_loss1 += loss1.item() * batch_targets.size(0)
        y_score = np.concatenate(all_logits)
        y_true = np.concatenate(test_targets)
        roc_auc = roc_auc_score(y_true, y_score)
        y_score1 = np.concatenate(all_logits1)
        roc_auc1 = roc_auc_score(y_true, y_score1)
        loss_diff.append(total_loss / total - total_loss1 / total)
        acc_diff.append(correct / total - correct1 / total)
        roc_diff.append(roc_auc - roc_auc1)
        print(f"Bootstrap iteration {_ + 1}: Loss difference: {loss_diff[-1]:.4f}, Accuracy difference: {acc_diff[-1]:.4f}, ROC AUC difference: {roc_diff[-1]:.4f}")
    return loss_diff, acc_diff, roc_diff

In [ ]:
loss_unsorted, acc_unsorted, roc_unsorted = bootstrap1('model_4_unsorted/epoch_12.pth', type='unsorted')

In [ ]:
loss_nofc, acc_nofc, roc_nofc = bootstrap1('model_4_nofc/epoch_14.pth', type='nofc')

In [ ]:
loss_nodist, acc_nodist, roc_nodist = bootstrap1('model_4/epoch_15.pth', type='nodist')

In [ ]:
loss_avg, acc_avg, roc_avg = bootstrap1('model_4/epoch_13.pth', type='avg')

In [ ]:
mean_loss_unsorted, mean_acc_unsorted, mean_roc_unsorted = np.mean(loss_unsorted), np.mean(acc_unsorted), np.mean(roc_unsorted)
mean_loss_nofc, mean_acc_nofc, mean_roc_nofc = np.mean(loss_nofc), np.mean(acc_nofc), np.mean(roc_nofc)
mean_loss_nodist, mean_acc_nodist, mean_roc_nodist = np.mean(loss_nodist), np.mean(acc_nodist), np.mean(roc_nodist)
mean_loss_avg, mean_acc_avg, mean_roc_avg = np.mean(loss_avg), np.mean(acc_avg), np.mean(roc_avg)
sd_loss_unsorted, sd_acc_unsorted, sd_roc_unsorted = np.std(loss_unsorted, ddof=1), np.std(acc_unsorted, ddof=1), np.std(roc_unsorted, ddof=1)
sd_loss_nofc, sd_acc_nofc, sd_roc_nofc = np.std(loss_nofc, ddof=1), np.std(acc_nofc, ddof=1), np.std(roc_nofc, ddof=1)
sd_loss_nodist, sd_acc_nodist, sd_roc_nodist = np.std(loss_nodist, ddof=1), np.std(acc_nodist, ddof=1), np.std(roc_nodist, ddof=1)
sd_loss_avg, sd_acc_avg, sd_roc_avg = np.std(loss_avg, ddof=1), np.std(acc_avg, ddof=1), np.std(roc_avg, ddof=1)